# TrainModel — PointPillars trên toàn bộ nuScenes TRAIN có sẵn (>50 GB LiDAR, A100)
Runtime **GPU A100** (Runtime → Change runtime type → A100, High-RAM). Chạy **Runtime → Run all**.

- Huấn luyện PointPillars **từ đầu** (from scratch, khởi tạo ngẫu nhiên, không nạp checkpoint có sẵn) trên toàn bộ 205 scene complete official TRAIN (8.182 keyframes, 10 sweeps, >50 GB LiDAR).
- Tập kiểm tra nội bộ (heldout evaluation subset): 15 scene complete official VAL ngoài pool cũ (602 keyframes).
- **Tập T lịch sử (benchmark pitch-v2) TUYỆT ĐỐI KHÔNG ĐỤNG ĐẾN.** Không có giao thoa scene hay sample_token nào với benchmark T.
- Tập V/P cũ trong pool đã được chuyển thành TRAIN trong hồ sơ 16, không gọi là holdout và không đánh giá lại.
- Gói dữ liệu nạp từ Drive: `MyDrive/datatest/trainval_pp_v2/` chứa các part ZIP độc lập ≤ 2 GB (`trainval_part_*.zip`), mã nguồn `code_trainmodel.zip` và `manifest.json`.
- Thư mục làm việc cục bộ: `/content/vcf_trainmodel`. Checkpoint được đồng bộ nguyên tử lên Drive mỗi 300 s (`MyDrive/vcf_results/trainmodel-full-20261014/`).
- Nếu ngắt runtime giữa chừng: kết nối lại A100 và **Run all** với cùng `RUN_ID`, tiến trình tự động khôi phục và tiếp tục từ checkpoint gần nhất.


In [ ]:
# @title 1 · Tham số, Drive, kiểm tra GPU A100, RAM, dung lượng đĩa
RUN_ID = 'trainmodel-full-20261014'   # @param {type:"string"} — giữ nguyên để resume cùng thư mục Drive
EPOCHS = 20                       # @param {type:"integer"} — đo A100: ~51 phút/epoch (CBGS 4327 it × 0,71 s)
BATCH = 8                         # @param {type:"integer"} — OOM thì ô 8 tự hạ 4 rồi 2
WORKERS = 8                       # @param {type:"integer"}
SWEEPS = 10                       # @param {type:"integer"} — 10 sweeps theo hợp đồng hồ sơ 16
SYNC_EVERY_S = 300                # chu kỳ đồng bộ checkpoint lên Drive (s)

import os, sys, json, time, hashlib, zipfile, shutil, subprocess, threading, re
from pathlib import Path
from datetime import datetime, timezone

T_NOTEBOOK = time.time()
def stamp(msg):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}Z] {msg}", flush=True)

from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = Path('/content/drive/MyDrive')

PACK_DIR = MYDRIVE / 'datatest/trainval_pp_v2'
if not (PACK_DIR / 'manifest.json').is_file():
    for cand in [MYDRIVE / 'vcf_pack/trainval_pp_v2', MYDRIVE / 'trainval_pp_v2']:
        if (cand / 'manifest.json').is_file():
            PACK_DIR = cand
            break
    if not (PACK_DIR / 'manifest.json').is_file():
        raise FileNotFoundError(f"Không tìm thấy manifest.json tại {PACK_DIR}. Cần upload gói trainval_pp_v2 lên MyDrive/datatest/trainval_pp_v2/")

MANIFEST = json.loads((PACK_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert MANIFEST.get('format') == 'vcf-trainval-full-v1', f"Manifest format không hợp lệ: {MANIFEST.get('format')}"
assert MANIFEST.get('sweeps') == 10, f"Manifest sweeps={MANIFEST.get('sweeps')} != 10"

RESULTS = MYDRIVE / 'vcf_results'
SAVE = RESULTS / RUN_ID
for sub in ('logs', 'seed', 'train/pcdet_output'):
    (SAVE / sub).mkdir(parents=True, exist_ok=True)

RUN = Path('/content/vcf_trainmodel')
REPO = RUN / 'repo'
DATA_ROOT = RUN / 'data'
EXP = RUN / 'exp'
VCF_HOME = RUN / 'vcf_home'
for d in (RUN, RUN / 'logs', RUN / 'markers', DATA_ROOT, EXP, VCF_HOME):
    d.mkdir(parents=True, exist_ok=True)
PY = sys.executable

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('Runtime không có GPU: Runtime → Change runtime type → A100')
GPU = gpu.stdout.strip()
if 'A100' not in GPU:
    raise RuntimeError(f'Cần GPU A100, đang phát hiện {GPU}. Runtime → Change runtime type → A100 + High-RAM')

mem_gb = int(open('/proc/meminfo').read().split('MemTotal:')[1].split()[0]) / 1e6
free_gb = shutil.disk_usage('/content').free / 1e9
source_bytes = MANIFEST.get('source_bytes', 61895634074)
unpacked = (RUN / 'markers' / 'data_unpacked').is_file()
req_free_gb = (35e9 if unpacked else source_bytes + 35e9) / 1e9   # dữ liệu đã giải nén thì chỉ cần dự phòng checkpoint

stamp(f"GPU: {GPU} · RAM: {mem_gb:.1f} GB · Đĩa trống: {free_gb:.1f} GB (yêu cầu: >= {req_free_gb:.1f} GB)")
if mem_gb < 40:
    raise RuntimeError(f'RAM {mem_gb:.1f} GB < 40 GB — bật High-RAM (Runtime → Change runtime type) rồi chạy lại')
if free_gb < req_free_gb:
    raise RuntimeError(f"Đĩa /content còn {free_gb:.1f} GB < yêu cầu {req_free_gb:.1f} GB")

stamp(f"Khởi tạo thành công: RUN_ID={RUN_ID} · EPOCHS={EPOCHS} · PACK_DIR={PACK_DIR} · SAVE={SAVE}")


In [ ]:
# @title 2 · Hàm đo lường, quản lý tiến trình, đánh dấu bước và đồng bộ Drive
assert 'SAVE' in globals(), 'Chạy ô 1 trước'
TIMING_PATH = SAVE / 'timing.json'
TIMING = json.loads(TIMING_PATH.read_text(encoding='utf-8'))['steps'] if TIMING_PATH.is_file() else []

def gpu_used_mib():
    try:
        out = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], text=True)
        return max(float(v) for v in out.split())
    except Exception:
        return None

def save_timing():
    tmp = TIMING_PATH.with_suffix('.part')
    tmp.write_text(json.dumps(dict(run_id=RUN_ID, gpu=GPU, steps=TIMING), indent=2), encoding='utf-8')
    os.replace(tmp, TIMING_PATH)

def measured(name, action):
    stop, samples = threading.Event(), []
    def monitor():
        while not stop.is_set():
            v = gpu_used_mib()
            if v is not None:
                samples.append(v)
            stop.wait(1.0)
    th = threading.Thread(target=monitor, daemon=True); th.start()
    t0, status = time.perf_counter(), 0
    stamp(f'▶ {name}')
    try:
        return action()
    except BaseException as exc:
        status = getattr(exc, 'returncode', 1)
        raise
    finally:
        stop.set(); th.join()
        TIMING.append(dict(step=name, elapsed_s=round(time.perf_counter() - t0, 1), exit_code=status,
                           gpu_peak_used_mib=max(samples) if samples else None,
                           gpu_measurement='nvidia-smi device-wide 1 s',
                           utc=datetime.now(timezone.utc).isoformat()))
        save_timing()
        stamp(f"◀ {name}: {TIMING[-1]['elapsed_s']} s · exit {status} · VRAM đỉnh {TIMING[-1]['gpu_peak_used_mib']} MiB")

def env():
    e = os.environ.copy()
    pcdet_root = VCF_HOME / 'OpenPCDet'
    e.update(PYTHONIOENCODING='utf-8', PYTHON=PY, VCF_HOME=str(VCF_HOME), PCDET_ROOT=str(pcdet_root),
             PYTHONPATH=f"{REPO / 'model/worker'}:{pcdet_root}")
    return e

def command(name, args, cwd=None, quiet=False):
    log = RUN / 'logs' / f'{name}.log'
    def run():
        with log.open('a', encoding='utf-8') as f:
            f.write(f'\n# {datetime.now(timezone.utc).isoformat()} $ ' + ' '.join(map(str, args)) + '\n')
            p = subprocess.Popen(list(map(str, args)), cwd=str(cwd or REPO / 'model/worker'), env=env(),
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
            last = []
            for line in p.stdout:
                f.write(line)
                last = (last + [line])[-40:]
                if not quiet:
                    print(line, end='')
            p.wait()
        shutil.copy2(log, SAVE / 'logs' / log.name)
        if p.returncode != 0:
            if quiet:
                print(''.join(last))
            raise subprocess.CalledProcessError(p.returncode, args)
        return ''.join(last)
    return measured(name, run)

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

def done(name, **info):
    (SAVE / 'logs' / f'{name}_done.json').write_text(
        json.dumps(dict(utc=datetime.now(timezone.utc).isoformat(), **info), indent=2, default=str),
        encoding='utf-8')

def is_done(name):
    return (SAVE / 'logs' / f'{name}_done.json').is_file()

def local_ok(name):
    return (RUN / 'markers' / name).is_file()

def mark_local(name):
    (RUN / 'markers' / name).write_text(datetime.now(timezone.utc).isoformat())

def to_drive(src, dst):
    # SHA lấy từ file NGUỒN; bản trên Drive phải khớp rồi mới ghi .sha256 (file cụt không thể tự 'verified')
    dst.parent.mkdir(parents=True, exist_ok=True)
    sha = digest(src)
    tmp = dst.with_name(dst.name + '.part')
    shutil.copy2(src, tmp); os.replace(tmp, dst)
    got = digest(dst)
    if got != sha:
        dst.unlink(missing_ok=True)
        raise IOError(f'Copy lên Drive hỏng {dst.name}: SHA đích {got[:12]} != nguồn {sha[:12]}')
    dst.with_name(dst.name + '.sha256').write_text(f'{sha}  {dst.name}\n')
    return sha

def verified(dst):
    s = dst.with_name(dst.name + '.sha256')
    return dst.is_file() and s.is_file() and s.read_text().split()[0] == digest(dst)

def extract_checked(zpath, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for item in z.infolist():
            dest = (target / item.filename).resolve()
            if not dest.is_relative_to(target.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP entry không an toàn: ' + item.filename)
        z.extractall(target)

stamp(f'Helpers sẵn sàng · {len(TIMING)} bước đã ghi nhận từ các phiên trước')


In [ ]:
# @title 3 · Giải nén và kiểm tra toàn vẹn mã nguồn (code_trainmodel.zip)
assert 'measured' in globals(), 'Chạy ô 2 trước'
CODE_INFO = MANIFEST.get('code')
assert CODE_INFO and 'name' in CODE_INFO, 'Manifest gói trainval_pp_v2 thiếu thông tin code archive'
CODE_ZIP = PACK_DIR / CODE_INFO['name']
assert CODE_ZIP.is_file(), f'Không tìm thấy {CODE_ZIP}'

mark = REPO / '.extracted_source_hash'
SRC_HASH = CODE_INFO['source_hash']
ZIP_SHA = CODE_INFO['sha256']

if mark.is_file() and mark.read_text().strip() == SRC_HASH:
    stamp(f'Mã nguồn {SRC_HASH[:12]} đã được giải nén — bỏ qua')
else:
    def unpack_code():
        stamp(f'Kiểm tra sha256 {CODE_ZIP.name}...')
        act_sha = digest(CODE_ZIP)
        assert act_sha == ZIP_SHA, f'sha256 code ZIP lệch manifest: thực tế {act_sha} != kỳ vọng {ZIP_SHA}'
        with zipfile.ZipFile(CODE_ZIP) as z:
            comment = z.comment.decode('utf-8', errors='replace').strip()
            assert comment == SRC_HASH, f'Comment trong code ZIP lệch source_hash: {comment} != {SRC_HASH}'
        shutil.rmtree(REPO, ignore_errors=True)
        extract_checked(CODE_ZIP, REPO)
        mark.write_text(SRC_HASH)
    measured('code_extract', unpack_code)

SCRIPT_FULL = REPO / 'model/scripts/trainval_full.py'
SETUP_SH = REPO / 'model/scripts/colab_setup.sh'
TRAIN_SEED_PY = REPO / 'model/worker/c4/lidar/tier1/train_seed.py'
assert SCRIPT_FULL.is_file(), f'Thiếu {SCRIPT_FULL}'
assert SETUP_SH.is_file(), f'Thiếu {SETUP_SH}'
assert TRAIN_SEED_PY.is_file(), f'Thiếu {TRAIN_SEED_PY}'

# Danh tính lượt chạy: SAVE (checkpoint, model cuối) chỉ thuộc đúng gói dữ liệu + mã + sweeps + epochs này
RUN_IDENTITY = dict(run_id=RUN_ID, dataset_manifest_sha256=digest(PACK_DIR / 'manifest.json'),
                    source_hash=SRC_HASH, sweeps=SWEEPS, epochs=EPOCHS)
IDENTITY_PATH = SAVE / 'run_identity.json'
if IDENTITY_PATH.is_file():
    old = json.loads(IDENTITY_PATH.read_text(encoding='utf-8'))
    if old != RUN_IDENTITY:
        diff = {k: (old.get(k), v) for k, v in RUN_IDENTITY.items() if old.get(k) != v}
        raise RuntimeError(f'{SAVE} thuộc lượt chạy khác {diff} — đổi RUN_ID để train lượt mới, '
                           'không dùng lại checkpoint/model của cấu hình cũ')
    stamp('Danh tính lượt chạy khớp thư mục Drive — được phép resume')
elif any((SAVE / 'seed').iterdir()) or any((SAVE / 'train/pcdet_output').rglob('*.pth')):
    raise RuntimeError(f'{SAVE} đã có kết quả nhưng thiếu run_identity.json — đổi RUN_ID')
else:
    IDENTITY_PATH.write_text(json.dumps(RUN_IDENTITY, indent=2), encoding='utf-8')

done('code', code_sha256=ZIP_SHA, source_hash=SRC_HASH)
stamp(f'MÃ NGUỒN HỢP LỆ · source_hash: {SRC_HASH}')


In [ ]:
# @title 4 · Cài đặt môi trường huấn luyện PointPillars (colab_setup.sh)
assert 'SRC_HASH' in globals(), 'Chạy ô 3 trước'
# Colab có file .gitignore trong dist-packages ⇒ pccm tưởng cumm là bản editable và JIT-build (thiếu header) ⇒ tắt JIT,
# dùng core_cc dựng sẵn của wheel. env() chép os.environ nên mọi tiến trình con (train, verify) đều nhận.
os.environ.update(CUMM_DISABLE_JIT='1', SPCONV_DISABLE_JIT='1')
INSTALL_MARK = Path(f'/tmp/.vcf_trainmodel_install_{SRC_HASH[:12]}')

if INSTALL_MARK.is_file():
    stamp('Môi trường PointPillars đã được cài trong phiên runtime này — bỏ qua')
else:
    def run_setup():
        command('colab_setup', ['bash', str(SETUP_SH)], cwd=REPO)
        INSTALL_MARK.write_text(SRC_HASH)
    try:
        measured('colab_setup', run_setup)
    except Exception as e:
        stamp(f'colab_setup lỗi: {e}. Thử lại lần thứ hai theo quy tắc SPEC-P01 §4...')
        measured('colab_setup_retry', run_setup)

# bước OpenPCDet của setup cài numba/scikit-image không ghim ⇒ kéo numpy 2.x; nuscenes-devkit + spconv cần numpy<2
command('pin_numpy', [PY, '-m', 'pip', 'install', '-q', 'numpy<2', 'numba==0.61.2', 'llvmlite==0.44.0'], quiet=True)

probe = ("import numpy, torch, spconv, pcdet; "
         "import pcdet.ops.iou3d_nms.iou3d_nms_cuda, pcdet.ops.roiaware_pool3d.roiaware_pool3d_cuda; "
         "from spconv.pytorch import SparseConv3d; "
         "print('numpy', numpy.__version__, '| torch', torch.__version__, torch.version.cuda, "
         "'| cuda', torch.cuda.is_available(), '| spconv', spconv.__version__, '| pcdet', pcdet.__version__)")
command('env_probe', [PY, '-c', probe])

train_py_path = VCF_HOME / 'OpenPCDet/tools/train.py'
assert train_py_path.is_file(), f'Không tìm thấy {train_py_path}'
train_py = train_py_path.read_text(encoding='utf-8')
RESUME_SUPPORTED = 'ckpt_list' in train_py
stamp(f'Kiểm tra OpenPCDet tools/train.py cơ chế resume: {RESUME_SUPPORTED} (bắt buộc True)')
assert RESUME_SUPPORTED, 'OpenPCDet tools/train.py thiếu cơ chế resume checkpoint tự động'

done('setup', resume_supported=RESUME_SUPPORTED)
stamp('MÔI TRƯỜNG POINTPILLARS ĐÃ SẴN SÀNG')


In [ ]:
# @title 5 · Kiểm tra SHA từng phần và giải nén dữ liệu nội bộ (/content/vcf_trainmodel/data) — đọc Drive 1 lượt
assert 'RESUME_SUPPORTED' in globals(), 'Chạy ô 4 trước'
PART_TMP = Path('/content/_part_tmp')

def copy_hashed(src, dst):
    # chép Drive -> đĩa local và tính SHA trong CÙNG một lượt đọc
    h = hashlib.sha256()
    with open(src, 'rb') as fi, open(dst, 'wb') as fo:
        for chunk in iter(lambda: fi.read(1 << 24), b''):
            h.update(chunk); fo.write(chunk)
    return h.hexdigest()

def unpack_trainval_parts():
    parts = MANIFEST['parts']
    total_parts = len(parts)
    PART_TMP.mkdir(parents=True, exist_ok=True)
    stamp(f'Bắt đầu kiểm tra và giải nén {total_parts} part ({MANIFEST["source_bytes"]/1e9:.2f} GB source)...')
    n_files = 0
    for idx, part in enumerate(parts, 1):
        part_name = part['name']
        archive = PACK_DIR / part_name
        mk = RUN / 'markers' / f'part_{part_name}'
        assert archive.is_file(), f'Thiếu file part: {archive}'
        assert archive.stat().st_size == part['size_bytes'], f'{part_name} sai kích thước: {archive.stat().st_size} != {part["size_bytes"]}'
        with zipfile.ZipFile(archive) as z:   # chỉ đọc central directory (vài MB)
            infos = [f for f in z.infolist() if not f.is_dir()]
        assert len(infos) == part['file_count'], f'{part_name} số lượng file lệch: {len(infos)} != {part["file_count"]}'
        if mk.is_file():   # marker chỉ ghi SAU khi part đã khớp SHA và giải nén xong
            n_files += len(infos)
            stamp(f"  [{idx}/{total_parts}] {part_name} đã giải nén trước đó — bỏ qua")
            continue
        local = PART_TMP / part_name
        act_sha = copy_hashed(archive, local)
        if act_sha != part['sha256']:
            local.unlink(missing_ok=True)
            raise ValueError(f'{part_name} sai SHA256: {act_sha} != {part["sha256"]}')
        extract_checked(local, RUN)   # zipfile tự kiểm CRC từng file khi giải nén
        local.unlink()
        mk.write_text(act_sha)
        n_files += len(infos)
        stamp(f"  [{idx}/{total_parts}] {part_name} OK ({len(infos)} files, SHA khớp)")

    assert n_files == MANIFEST.get('file_count', len(MANIFEST['files'])), f"Tổng số file giải nén {n_files} lệch manifest"
    return DATA_ROOT

if local_ok('data_unpacked') and (DATA_ROOT / 'v1.0-trainval/scene.json').is_file() and (DATA_ROOT / 'samples/LIDAR_TOP').is_dir() and (DATA_ROOT / 'sweeps/LIDAR_TOP').is_dir():
    stamp('Dữ liệu trainval đã được giải nén và kiểm tra trước đó — bỏ qua')
else:
    measured('data_unpack', unpack_trainval_parts)
    assert (DATA_ROOT / 'v1.0-trainval/scene.json').is_file(), 'Thiếu v1.0-trainval/scene.json sau khi giải nén'
    assert (DATA_ROOT / 'maps').is_dir(), 'Thiếu maps/ sau khi giải nén'
    assert (DATA_ROOT / 'samples/LIDAR_TOP').is_dir(), 'Thiếu samples/LIDAR_TOP sau khi giải nén'
    assert (DATA_ROOT / 'sweeps/LIDAR_TOP').is_dir(), 'Thiếu sweeps/LIDAR_TOP sau khi giải nén'
    mark_local('data_unpacked')

n_kf = sum(1 for _ in (DATA_ROOT / 'samples/LIDAR_TOP').iterdir())
n_sw = sum(1 for _ in (DATA_ROOT / 'sweeps/LIDAR_TOP').iterdir())
stamp(f'DỮ LIỆU SẴN SÀNG · Keyframes: {n_kf} · Sweeps: {n_sw} · Thư mục: {DATA_ROOT}')
done('data', keyframes=n_kf, sweeps=n_sw)


In [ ]:
# @title 6 · Kiểm tra độ phủ metadata, tạo chỉ mục EXP/index.parquet và cổng phân vùng
assert 'DATA_ROOT' in globals() and (DATA_ROOT / 'v1.0-trainval').is_dir(), 'Chạy ô 5 trước'

command('trainval_verify', [PY, str(SCRIPT_FULL), 'verify', '--data-root', str(DATA_ROOT),
                            '--manifest', str(PACK_DIR / 'manifest.json')], cwd=REPO)

command('trainval_index', [PY, str(SCRIPT_FULL), 'index', '--data-root', str(DATA_ROOT),
                           '--manifest', str(PACK_DIR / 'manifest.json'), '--exp', str(EXP)], cwd=REPO)

INDEX_PATH = EXP / 'index.parquet'
assert INDEX_PATH.is_file(), f'Không tìm thấy {INDEX_PATH}'

import pandas as pd
idx_df = pd.read_parquet(INDEX_PATH)
n_train = int((idx_df.split == 'S').sum())
n_eval = int((idx_df.split == 'T').sum())
stamp(f'Chỉ mục phân vùng: S (official TRAIN) = {n_train} frames | T (heldout VAL) = {n_eval} frames')
assert n_train == 8182, f'Số frame train S={n_train} != 8182 (hợp đồng 205 scene complete TRAIN)'
assert n_eval == 602, f'Số frame eval T={n_eval} != 602 (hợp đồng 15 scene complete VAL ngoài pool)'

excluded_t = set(MANIFEST.get('excluded_T_scenes', []))
index_scenes = set(idx_df.scene_name)
overlap = index_scenes & excluded_t
assert not overlap, f'VI PHẠM BẤT BIẾN: Chỉ mục chứa {len(overlap)} scene benchmark T cũ: {overlap}'

to_drive(INDEX_PATH, SAVE / 'seed/index.parquet')
to_drive(EXP / 'dataset_manifest_sha256.txt', SAVE / 'seed/dataset_manifest_sha256.txt')
shutil.copy2(PACK_DIR / 'manifest.json', SAVE / 'manifest.json')

INDEX_SHA = digest(INDEX_PATH)
done('index', train_frames=n_train, eval_frames=n_eval, index_sha256=INDEX_SHA,
     dataset_manifest_sha256=digest(PACK_DIR / 'manifest.json'))
stamp(f'INDEX VÀ CỔNG PHÂN VÙNG HỢP LỆ · SHA256: {INDEX_SHA}')


In [ ]:
# @title 7 · Khởi động huấn luyện nền PointPillars và luồng đồng bộ checkpoint lên Drive (300 s)
assert 'INDEX_SHA' in globals(), 'Chạy ô 6 trước'

SEED_SAVE = SAVE / 'seed'
TRAIN_SAVE = SAVE / 'train/pcdet_output'
PCDET_OUT = EXP / 't1/pcdet_output'
TRAIN_LOG = RUN / 'logs/train.log'

SEED_FILES = {'ckpt/seed_latest.pth': 'seed_latest.pth', 'cfg/pp_seed.yaml': 'pp_seed.yaml',
              'train_config.yaml': 'train_config.yaml', 'seed_tokens.txt': 'seed_tokens.txt'}

def seed_on_drive():
    # model cuối chỉ được dùng lại khi đủ file đúng SHA VÀ cùng danh tính lượt chạy
    ident = SEED_SAVE / 'run_identity.json'
    if not (all(verified(SEED_SAVE / d) for d in SEED_FILES.values()) and verified(ident)):
        return False
    return json.loads(ident.read_text(encoding='utf-8')) == RUN_IDENTITY

def epoch_of(p):
    m = re.search(r'checkpoint_epoch_(\d+)\.pth$', p.name)
    return int(m.group(1)) if m else -1

def restore_seed():
    for rel, d in SEED_FILES.items():
        dst = EXP / 't1' / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_file() and (SEED_SAVE / d).is_file():
            shutil.copy2(SEED_SAVE / d, dst)

def mirror_ckpts(final=False):
    if not PCDET_OUT.is_dir():
        return 0
    n, now = 0, time.time()
    all_ckpts = sorted(PCDET_OUT.rglob('checkpoint_epoch_*.pth'))
    for f in all_ckpts:
        rel = f.relative_to(PCDET_OUT)
        dst = TRAIN_SAVE / rel
        if (final or now - f.stat().st_mtime > 60) and (not dst.is_file() or dst.stat().st_size != f.stat().st_size
                                                        or dst.stat().st_mtime < f.stat().st_mtime):
            to_drive(f, dst)
            n += 1
            
    # giữ 2 epoch mới nhất theo SỐ (sắp chữ đặt epoch_10 trước epoch_2 ⇒ xoá nhầm bản mới)
    drive_ckpts = sorted(TRAIN_SAVE.rglob('checkpoint_epoch_*.pth'), key=epoch_of)
    if len(drive_ckpts) > 2:
        for old_ckpt in drive_ckpts[:-2]:
            old_ckpt.unlink(missing_ok=True)
            old_ckpt.with_name(old_ckpt.name + '.sha256').unlink(missing_ok=True)

    if TRAIN_LOG.is_file():
        shutil.copy2(TRAIN_LOG, SAVE / 'logs/train.log')
    return n

def restore_ckpts():
    n = 0
    for g in TRAIN_SAVE.rglob('checkpoint_epoch_*.pth') if TRAIN_SAVE.is_dir() else []:
        if not verified(g):
            stamp(f'CẢNH BÁO: Checkpoint {g.name} trên Drive sai SHA256 — bỏ qua khôi phục')
            continue
        dst = PCDET_OUT / g.relative_to(TRAIN_SAVE)
        if not dst.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(g, dst)
            n += 1
    return n

def train_running():
    return subprocess.run(['pgrep', '-f', 'c4.lidar.tier1.train_seed'], capture_output=True).returncode == 0

def start_train(batch):
    global TRAIN_PROC, TRAIN_T0, TRAIN_BATCH
    TRAIN_BATCH, TRAIN_T0 = batch, time.time()
    f = TRAIN_LOG.open('a', encoding='utf-8')
    args = [PY, '-m', 'c4.lidar.tier1.train_seed', '--exp', str(EXP), '--nusc', str(DATA_ROOT),
            '--sweeps', str(SWEEPS), '--epochs', str(EPOCHS), '--batch', str(batch), '--workers', str(WORKERS)]
    f.write(f'\n# {datetime.now(timezone.utc).isoformat()} $ ' + ' '.join(map(str, args)) + '\n'); f.flush()
    TRAIN_PROC = subprocess.Popen(args, cwd=str(REPO / 'model/worker'), env=env(), stdout=f,
                                  stderr=subprocess.STDOUT)
    stamp(f'Tiến trình train nền PID {TRAIN_PROC.pid} · batch {batch} · epochs {EPOCHS} · sweeps {SWEEPS} · log: {TRAIN_LOG}')

if globals().get('SYNC_STOP') is not None:
    globals()['SYNC_STOP'].set()
SYNC_STOP = threading.Event()

def sync_loop():
    while not SYNC_STOP.wait(SYNC_EVERY_S):
        try:
            k = mirror_ckpts()
            if k:
                stamp(f'Đã đồng bộ {k} checkpoint ổn định lên Drive')
        except Exception as e:
            print('Lỗi đồng bộ checkpoint (thử lại chu kỳ kế tiếp):', e)

TRAIN_PROC, TRAIN_BATCH, TRAIN_T0 = None, BATCH, time.time()
if seed_on_drive():
    restore_seed()
    stamp('Đã có đầy đủ mô hình hoàn thiện trên Drive (SHA hợp lệ) — BỎ QUA train')
else:
    if train_running():
        stamp('train_seed đang chạy từ phiên trước — tiếp tục theo dõi, ô 8 sẽ chờ')
    else:
        k = restore_ckpts()
        if k:
            stamp(f'Khôi phục {k} checkpoint từ Drive ⇒ OpenPCDet tự động resume (resume={RESUME_SUPPORTED})')
        start_train(BATCH)
    threading.Thread(target=sync_loop, daemon=True).start()


In [ ]:
# @title 8 · Theo dõi tiến trình huấn luyện, tính ETA sau Epoch 1 và cơ chế phục hồi OOM
assert 'PCDET_OUT' in globals(), 'Chạy ô 7 trước'

def epochs_done():
    eps = [int(m.group(1)) for f in PCDET_OUT.rglob('checkpoint_epoch_*.pth')
           if (m := re.search(r'checkpoint_epoch_(\d+)\.pth$', f.name))] if PCDET_OUT.is_dir() else []
    return max(eps) if eps else 0

def heartbeat():
    k = epochs_done()
    el = time.time() - TRAIN_T0
    eta = (el / k * (EPOCHS - k)) if k >= 1 else None
    tail = TRAIN_LOG.read_text(encoding='utf-8', errors='replace')[-600:].replace('\r', '\n').strip().splitlines()
    last_line = tail[-1][:160] if tail else ''
    stamp(f"Epoch {k}/{EPOCHS} · Đã chạy {el/60:.1f} phút · ETA {'chờ sau epoch 1' if eta is None else f'{eta/60:.1f} phút'} · "
          f"VRAM {gpu_used_mib()} MiB · {last_line}")

if not seed_on_drive():
    t_wait = time.time()
    while True:
        while (TRAIN_PROC is not None and TRAIN_PROC.poll() is None) or (TRAIN_PROC is None and train_running()):
            heartbeat(); time.sleep(60)
        
        rc = TRAIN_PROC.returncode if TRAIN_PROC is not None else (0 if (EXP / 't1/ckpt/seed_latest.pth').is_file() else 1)
        log_tail = TRAIN_LOG.read_text(encoding='utf-8', errors='replace')[-6000:]
        
        low = log_tail.lower()
        is_oom = rc != 0 and ('cuda out of memory' in low or 'outofmemoryerror' in low)
        if is_oom and TRAIN_BATCH > 2:
            mirror_ckpts(final=True)
            new_batch = TRAIN_BATCH // 2
            stamp(f'CUDA OOM tại batch {TRAIN_BATCH} ⇒ hạ batch xuống {new_batch} và resume từ checkpoint cuối')
            start_train(new_batch)
            continue
        elif rc != 0 and not is_oom:
            print(log_tail)
            raise RuntimeError(f'train_seed gặp lỗi exit {rc} (KHÔNG phải OOM) — xem {TRAIN_LOG}')
        break
    
    mirror_ckpts(final=True)
    SYNC_STOP.set()
    TIMING.append(dict(step='train_seed', elapsed_s=round(time.time() - t_wait, 1), exit_code=rc,
                       batch=TRAIN_BATCH, epochs=EPOCHS, sweeps=SWEEPS))
    save_timing()
    
    if rc != 0 or not (EXP / 't1/ckpt/seed_latest.pth').is_file():
        print(log_tail)
        raise RuntimeError(f'Huấn luyện không thành công (exit {rc}) — kiểm tra {TRAIN_LOG}')
    
    for rel, d in SEED_FILES.items():
        to_drive(EXP / 't1' / rel, SEED_SAVE / d)
    (EXP / 't1/run_identity.json').write_text(json.dumps(RUN_IDENTITY, indent=2), encoding='utf-8')
    to_drive(EXP / 't1/run_identity.json', SEED_SAVE / 'run_identity.json')

SYNC_STOP.set()
TRAIN_CFG_TEXT = (SEED_SAVE / 'train_config.yaml').read_text(encoding='utf-8')
print("=== train_config.yaml ===")
print(TRAIN_CFG_TEXT)
assert 'gt_sampling: false' in TRAIN_CFG_TEXT.lower(), 'BẤT BIẾN VI PHẠM: gt_sampling phải là False'
assert f'sweeps: {SWEEPS}' in TRAIN_CFG_TEXT, f'BẤT BIẾN VI PHẠM: sweeps phải là {SWEEPS}'
assert f'epochs: {EPOCHS}' in TRAIN_CFG_TEXT, f'BẤT BIẾN VI PHẠM: epochs phải là {EPOCHS}'

CKPT_SHA = digest(SEED_SAVE / 'seed_latest.pth')
assert CKPT_SHA == (SEED_SAVE / 'seed_latest.pth.sha256').read_text().split()[0], 'seed_latest.pth trên Drive lệch .sha256'
done('train', ckpt_sha256=CKPT_SHA, epochs=EPOCHS, sweeps=SWEEPS, batch=TRAIN_BATCH if 'TRAIN_BATCH' in globals() else BATCH)
stamp(f'HUẤN LUYỆN HOÀN TẤT · Checkpoint SHA256: {CKPT_SHA}')


In [ ]:
# @title 8b · (Tuỳ chọn) Dừng sớm: bấm ⏹ ô 8, đặt STOP_EARLY=True rồi chạy ô này → ô 9 → ô 10
STOP_EARLY = False                # @param {type:"boolean"}
assert STOP_EARLY, 'Chỉ chạy khi cố ý dừng sớm (đã ⏹ ô 8): đặt STOP_EARLY=True'
subprocess.run(['pkill', '-f', 'c4.lidar.tier1.train_seed']); subprocess.run(['pkill', '-f', 'train.py --cfg_file'])
time.sleep(5)
mirror_ckpts(final=True); SYNC_STOP.set()
ck = max(PCDET_OUT.rglob('checkpoint_epoch_*.pth'), key=epoch_of); E_DONE = epoch_of(ck)
stamp(f'Dùng {ck.name} (epoch {E_DONE}/{EPOCHS})')
r = subprocess.run([PY, '-m', 'c4.lidar.tier1.train_seed', '--exp', str(EXP), '--nusc', str(DATA_ROOT),
                    '--sweeps', str(SWEEPS), '--epochs', str(EPOCHS), '--batch', str(TRAIN_BATCH),
                    '--workers', str(WORKERS), '--init-ckpt', str(ck)],
                   cwd=str(REPO / 'model/worker'), env=env(), capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, f'train_seed --init-ckpt exit {r.returncode}'
for rel, d in SEED_FILES.items():
    to_drive(EXP / 't1' / rel, SEED_SAVE / d)
ident = dict(RUN_IDENTITY, epochs_trained=E_DONE,
             note='dừng sớm: LR one-cycle chưa giảm hết, chưa đánh giá VAL')
(EXP / 't1/run_identity.json').write_text(json.dumps(ident, indent=2, ensure_ascii=False), encoding='utf-8')
to_drive(EXP / 't1/run_identity.json', SEED_SAVE / 'run_identity.json')
CKPT_SHA = digest(SEED_SAVE / 'seed_latest.pth')
EPOCHS = E_DONE
done('train', ckpt_sha256=CKPT_SHA, epochs=E_DONE, sweeps=SWEEPS, batch=TRAIN_BATCH)
stamp(f'Đã xuất bản model epoch {E_DONE} · SHA256 {CKPT_SHA} · {(SEED_SAVE / "seed_latest.pth").stat().st_size/1e6:.1f} MB')


In [ ]:
# @title 9 · Xuất bản mô hình, kiểm tra SHA256SUMS và ghi nhận kết quả hoàn tất
assert 'CKPT_SHA' in globals(), 'Chạy ô 8 trước'

def sha_all():
    lines = []
    for f in sorted(SAVE.rglob('*')):
        if f.is_file() and not f.name.endswith(('.part', '.sha256')) and f.name not in ('SHA256SUMS.txt',):
            lines.append(f'{digest(f)}  {f.relative_to(SAVE).as_posix()}')
    out_path = SAVE / 'SHA256SUMS.txt'
    out_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return len(lines)

import yaml
train_cfg = yaml.safe_load((SEED_SAVE / 'train_config.yaml').read_text(encoding='utf-8'))
post_eval = train_cfg.get('post_train_eval', 'unknown')
n_seed = train_cfg.get('n_seed', 0)

parameters = dict(
    run_id=RUN_ID,
    model_type="PointPillars",
    dataset_format=MANIFEST.get('format'),
    dataset_manifest_sha256=digest(PACK_DIR / 'manifest.json'),
    source_hash=MANIFEST.get('code', {}).get('source_hash'),
    code_sha256=MANIFEST.get('code', {}).get('sha256'),
    index_sha256=INDEX_SHA,
    ckpt_sha256=CKPT_SHA,
    epochs=EPOCHS,
    batch=TRAIN_BATCH if 'TRAIN_BATCH' in globals() else BATCH,
    workers=WORKERS,
    sweeps=SWEEPS,
    gpu=GPU,
    n_seed_frames=n_seed,
    post_train_eval=post_eval,
    eval_note="post_train_eval ghi nhận trung thực theo nuScenes devkit subset (SPEC-P01 §3); không tuyên bố full-val mAP/NDS",
    created_utc=datetime.now(timezone.utc).isoformat(),
)
(SAVE / 'parameters.json').write_text(json.dumps(parameters, indent=2), encoding='utf-8')

run_manifest = dict(
    format="vcf-run-output-v1",
    run_id=RUN_ID,
    parameters=parameters,
    artifacts=dict(
        checkpoint="seed/seed_latest.pth",
        config="seed/pp_seed.yaml",
        train_config="seed/train_config.yaml",
        seed_tokens="seed/seed_tokens.txt",
        index="seed/index.parquet",
        sha256sums="SHA256SUMS.txt",
    ),
    steps=TIMING,
)
(SAVE / 'manifest.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

save_timing()
n_hashed = sha_all()   # băm SAU CÙNG: manifest.json/timing.json đã ghi xong

total_gb = sum(f.stat().st_size for f in SAVE.rglob('*') if f.is_file()) / 1e9
total_h = (time.time() - T_NOTEBOOK) / 3600
stamp('=================================================================')
stamp('🎉 TOÀN BỘ TIẾN TRÌNH HUẤN LUYỆN VÀ XUẤT BẢN ĐÃ HOÀN TẤT THÀNH CÔNG!')
stamp(f'RUN_ID: {RUN_ID} · Model Checkpoint: {CKPT_SHA} · post_train_eval: {post_eval}')
stamp(f'{n_hashed} files được băm SHA256 · Tổng dung lượng Drive: {total_gb:.2f} GB tại {SAVE}')
stamp(f'Tổng thời gian phiên: {total_h:.2f} giờ')
stamp('=================================================================')


In [ ]:
# @title 10 · Trả runtime A100 khi xong để ngừng tốn compute unit (kết quả đã ở Drive)
assert (SAVE / 'parameters.json').is_file() and (SAVE / 'SHA256SUMS.txt').is_file(), 'Ô 9 chưa xuất bản — KHÔNG trả runtime'
stamp(f'Đã xuất bản tại {SAVE} — trả runtime')
from google.colab import runtime
runtime.unassign()
